# A1.3 — Comparer le train et le test

Le test **visible** (`test.parquet`) est un échantillon tiré du train ; il sera remplacé par le test caché
lors de l'évaluation Kaggle (environ 1 500 spectres, 400 molécules, 1 à 16 spectres par molécule, médiane 3).
Les organisateurs l'ont construit pour ressembler au test caché : comprendre *comment* ils l'ont tiré nous dit
à quoi notre validation doit ressembler.

1. Retrouver d'où viennent les spectres du test visible.
2. Comparer le test, le train complet et la bibliothèque d'origine.
3. En tirer des recommandations pour la validation (C) et le nettoyage (A1.4).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from casmi.io import load_meta, load_spectra, load_test
from casmi.mass import compute_neutral_masses
from casmi.missing import ce_summary, instrument_family
from casmi.split import load_split

pd.set_option("display.width", 200)

meta = load_meta()
test = load_test()
test["num_peaks"] = test["ms2_mzs"].map(len)

## 1. D'où viennent les spectres du test visible ?

On cherche, pour chaque spectre du test, les spectres du train qui ont le même adduct, le même `precursor_mz`
(à 4 décimales) et le même nombre de pics, puis on compare les pics un à un.

In [ ]:
keys = ["adduct", "mz_key", "num_peaks"]
cand = (
    test.assign(mz_key=test["precursor_mz"].round(4))[["spectrum_id", "molecule_id", *keys]]
    .merge(meta.assign(mz_key=meta["precursor_mz"].round(4))[["row_id", "inchikey14", "ingest_lib", *keys]], on=keys)
)
peaks = load_spectra(cand["row_id"].unique()).set_index("row_id")
t = test.set_index("spectrum_id")
cand["mêmes pics"] = [
    np.allclose(t.at[s, "ms2_mzs"], peaks.at[r, "ms2_mzs"])
    and np.allclose(t.at[s, "ms2_normalized_intensities"], peaks.at[r, "ms2_normalized_intensities"])
    for s, r in zip(cand["spectrum_id"], cand["row_id"])
]
match = cand[cand["mêmes pics"]].drop_duplicates("spectrum_id")
print(f"Spectres du test retrouvés à l'identique dans le train : {len(match):,} / {len(test):,}")
print("Bibliothèque d'origine :", match["ingest_lib"].value_counts().to_dict())
print("Un seul inchikey14 par molecule_id :", (match.groupby("molecule_id")["inchikey14"].nunique() == 1).all())

**Tous les spectres du test visible sont des copies exactes de spectres d'enveda-180** (les timsTOF d'Enveda).
Conséquence immédiate : ses molécules sont dans le train. Un score calculé sur le test visible ne veut rien dire.

In [ ]:
split = load_split().set_index("inchikey14")
split.loc[match["inchikey14"].unique(), "split"].value_counts().rename("molécules du test visible")

### Quels spectres d'enveda-180 ont été retenus ?

Pour les 400 molécules du test, on compare les spectres retenus et ceux qui ne l'ont pas été.

In [ ]:
env = meta[meta["ingest_lib"] == "enveda-180"]
mol = env[env["inchikey14"].isin(match["inchikey14"])].assign(retenu=lambda d: d["row_id"].isin(match["row_id"]))
pd.crosstab(mol["adduct"], mol["retenu"]).sort_values(True, ascending=False)

Les adducts **dimères** (`[2M+Na]+`, `[2M+H]+`…) ne sont jamais retenus. Parmi les adducts simples, environ
2 spectres sur 3 le sont : la sélection ramène chaque molécule à quelques spectres.

## 2. Comparaison des distributions

Trois populations : le **train** complet, **enveda-180** (la bibliothèque d'origine du test) et le **test** visible.

In [ ]:
TEST_ADDUCTS = sorted(test["adduct"].unique())
groups = {"train": meta, "enveda-180": env, "test": test}
COLORS = {"train": "#2a78d6", "enveda-180": "#eb6834", "test": "#1baf7a"}  # palette catégorielle, ordre fixe

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.edgecolor": "#9c9b93", "axes.labelcolor": "#3c3b37",
    "xtick.color": "#5f5e58", "ytick.color": "#5f5e58", "font.size": 10, "legend.frameon": False,
})


def summary(d):
    cs = ce_summary(d["collision_energy_ev"])
    per_mol = d.groupby("molecule_id" if "molecule_id" in d else "inchikey14").size()
    return {
        "spectres": len(d),
        "molécules": len(per_mol),
        "spectres par molécule (médiane)": per_mol.median(),
        "spectres par molécule (max)": per_mol.max(),
        "% mode positif": round((d["ionization_mode"] == "positive").mean() * 100, 1),
        "% timsTOF": round((instrument_family(d["instrument_type"]) == "timsTOF").mean() * 100, 1),
        "nombre d'adducts": d["adduct"].nunique(),
        "% adducts du test": round(d["adduct"].isin(TEST_ADDUCTS).mean() * 100, 1),
        "precursor_mz (médiane)": round(d["precursor_mz"].median(), 1),
        "precursor_mz (90e centile)": round(d["precursor_mz"].quantile(0.9), 1),
        "pics par spectre (médiane)": d["num_peaks"].median(),
        "% spectres < 10 pics": round((d["num_peaks"] < 10).mean() * 100, 1),
        "% avec 3 énergies ou plus": round((cs["n_ce"] >= 3).mean() * 100, 1),
        "% sans énergie": round((cs["n_ce"] == 0).mean() * 100, 1),
        "log10 base_peak_intensity (médiane)": round(np.log10(d["base_peak_intensity"]).median(), 2),
    }


pd.DataFrame({name: summary(d) for name, d in groups.items()})

### Figure 1 — Les adducts

Part des spectres pour les adducts les plus fréquents de chaque population.

In [ ]:
top = pd.Index(meta["adduct"].value_counts().index[:8]).union(TEST_ADDUCTS, sort=False)
shares = pd.DataFrame({name: d["adduct"].value_counts(normalize=True).reindex(top).fillna(0) * 100 for name, d in groups.items()})
shares = shares.sort_values("train")

fig, ax = plt.subplots(figsize=(8, 5.5))
height = 0.26
for i, name in enumerate(groups):
    ax.barh(np.arange(len(shares)) + (1 - i) * height, shares[name], height=height * 0.92, color=COLORS[name], label=name)
ax.set_yticks(np.arange(len(shares)), shares.index)
ax.set_xlabel("% des spectres")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
ax.set_title("Le test ne contient que des adducts simples", loc="left", fontsize=12)
plt.tight_layout()

### Figure 2 — Le nombre de pics par spectre

Courbe cumulée : pour chaque nombre de pics N, la part des spectres qui en ont moins de N.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, d in groups.items():
    x = np.sort(d["num_peaks"].to_numpy())
    ax.step(x, np.arange(1, len(x) + 1) / len(x) * 100, where="post", linewidth=2, color=COLORS[name], label=name)
ax.axvline(10, color="#9c9b93", linewidth=1, linestyle="--")
ax.annotate("10 pics", (10, 95), xytext=(4, 0), textcoords="offset points", color="#5f5e58", fontsize=9)
ax.set_xscale("log")
ax.set_xlim(1, 1e4)
ax.set_ylim(0, 100)
ax.set_xlabel("nombre de pics N (échelle log)")
ax.set_ylabel("% des spectres ayant moins de N pics")
ax.legend(loc="lower right")
ax.set_title("Les spectres du test sont bien plus riches que ceux du train", loc="left", fontsize=12)
plt.tight_layout()

### La masse des molécules

Masse neutre calculée par la fonction de B (`compute_neutral_masses`), médiane par molécule.

In [ ]:
def molecule_masses(d, key):
    return d.assign(m=compute_neutral_masses(d)).groupby(key)["m"].median().dropna()


masses = {"train": molecule_masses(meta, "inchikey14"), "enveda-180": molecule_masses(env, "inchikey14"),
          "test": molecule_masses(test, "molecule_id")}
lo, hi = masses["test"].min(), masses["test"].max()
print(f"Masses des molécules du test : de {lo:.0f} à {hi:.0f} Da")
for name, m in masses.items():
    print(f"  {name:11s} médiane {m.median():.0f} Da, dans la plage du test : {m.between(lo, hi).mean():.1%}")

La plage de masse du test est celle d'enveda-180 (99,4 % de ses molécules y sont). Le train complet y est aussi
en grande partie (86 %), avec une queue de molécules plus lourdes.

### Figure 3 — Le nombre de spectres par molécule

C'est ce qui distingue le plus une requête du test d'une molécule du train : le test garde peu de spectres par molécule.

In [ ]:
def spectra_per_molecule(d, key, cap=12):
    n = d.groupby(key).size().clip(upper=cap)
    return n.value_counts(normalize=True).reindex(range(1, cap + 1), fill_value=0) * 100


CAP = 12
counts = pd.DataFrame({
    "train": spectra_per_molecule(meta, "inchikey14", CAP),
    "enveda-180": spectra_per_molecule(env, "inchikey14", CAP),
    "test": spectra_per_molecule(test, "molecule_id", CAP),
})
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.27
for i, name in enumerate(counts):
    ax.bar(counts.index + (i - 1) * width, counts[name], width=width * 0.92, color=COLORS[name], label=name)
ax.set_xticks(counts.index, [str(k) for k in counts.index[:-1]] + [f"{CAP}+"])
ax.set_xlabel("nombre de spectres de la molécule")
ax.set_ylabel("% des molécules")
ax.grid(axis="x", visible=False)
ax.legend()
ax.set_title("Une molécule du test a 1 à 9 spectres, 3 en médiane", loc="left", fontsize=12)
plt.tight_layout()

## 3. Ce qui est pareil, ce qui diffère

| | Test visible vs enveda-180 | Test visible vs train complet |
|---|---|---|
| Bibliothèque, instrument | identique (100 % enveda-180, timsTOF) | 45 % de timsTOF dans le train |
| Adducts | sans les dimères `[2M+…]` (29 % d'enveda-180) | 121 adducts dans le train, 7 dans le test |
| Masse des molécules | même plage (246 à 439 Da) | 86 % du train dans cette plage, plus une queue de molécules lourdes |
| Pics par spectre | légèrement plus (adducts simples) | 5 fois plus (médiane 230 contre 42) |
| Énergies de collision | identiques (20, 40, 60 eV ; 1 ou 3 énergies) | 13 % de spectres sans énergie dans le train |
| Spectres par molécule | environ la moitié (médiane 3 contre 6) | médiane 3 contre 6 |
| Valeurs manquantes | aucune | voir A1.2 |

Le test visible ressemble donc à **« enveda-180, adducts simples, environ 3 spectres par molécule »**. On ne sait
pas si le test caché suit les mêmes règles, mais c'est l'hypothèse la plus probable : les organisateurs l'ont
construit pour lui ressembler.

## 4. Recommandations

**Pour la validation (C, `build_queries`) :**
1. Garder le split tel quel, mais rapporter deux scores : sur toute la validation, et sur des **requêtes « type
   test »** : spectres d'enveda-180, adducts du test uniquement.
2. Plafonner le nombre de spectres par requête en imitant le test : tirer, avec une graine fixe, un nombre de
   spectres suivant la distribution du test (1 à 9, médiane 3).
3. Ne jamais évaluer sur le test visible : ses spectres sont des copies du train.

**Pour le nettoyage (A1.4) :**
4. Les spectres de très peu de pics (moins de 10 : 18 % du train, 2 % du test) et les adducts
   dimères ne ressemblent pas au test. Ce sont des candidats à écarter, ou à garder et tester en A4.1, plutôt
   qu'à supprimer d'office.

**Pour le filtre par masse (B) :**
5. timsTOF est très précis (99 % des erreurs sous 4,4 ppm) : une tolérance qui dépend de l'instrument est à tester.